## Setup

In [1]:
# Put all imports here #######################
import sys, json, glob
from pathlib import Path
from typing import Optional, Sequence
import numpy as np
import torch
from rdkit import Chem
from IPython.display import IFrame, display
import pubchempy as pcp
# NOTE: the MolPLAtte imports come AFTER sys.path is set below -- importing
# them here would fail on a fresh kernel.
##############################################

SRC = next((p / "src" for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "lead_optimization.py").is_file()), None)
if SRC is None:
    raise RuntimeError(
        "cannot find lead_optimization.py -- run this notebook from inside the "
        "MolPLAtte repo, or set SRC to <repo>/molplatte/src by hand")
sys.path.insert(0, str(SRC))
print("src:", SRC)


from lead_optimization import LeadOptimizer, pocket_embedding_from_structure
from lead_report import (run_lead_optimization, PERMISSIBLE_FLAVORS,
                         TASTE_FLAVORS, ODOUR_FLAVORS)
from html_report import render_html

DATA       = Path.home() / "preprocessed" / "molplatte"
# v2, 2026-09-15: width 512, lr 1.0e-4, tau 0.01, no freezing.
# H@1 0.3648 against the width-300 predecessor's 0.3251 (+12%), and
# novel@10 0.4251 vs 0.3445 (+23%) -- retrieval of R-groups absent from the
# training vocabulary, the metric nothing else moved.
# Width and lr had never been swept; both came from MolPLA's release.
CHECKPOINT = Path.home() / "checkpoints" / "molplatte" / "molplatte-final-v2-w512.pt"
VOCAB      = DATA / "union_vocab" / "base-full__crossdocked__tastepocket" / "rgroup_vocab.pkl.gz"

opt = LeadOptimizer.load(CHECKPOINT, VOCAB, device="cuda")
opt.build_library(batch_size=1024)

print(f"library rows      {len(opt.vocab):,}")
print(f"effective size    {opt.vocab.effective_size():.0f}")
print(f"top-1 share       {100 * opt.vocab.frequency_prior[0]:.2f}%")
print(f"novel rows        {len(opt._novel):,}  (absent from the pretraining vocabulary)")

# The permissible flavour strings -- the 24 condvec bits the corpus stores.
print(f"\nflavours ({len(PERMISSIBLE_FLAVORS)}):")
print("  taste:", ", ".join(TASTE_FLAVORS))
print("  odour:", ", ".join(ODOUR_FLAVORS))
print("  other: odorless, unknown")

src: /home/mogan/github/MolPLAtte/molplatte/src
library rows      91,935
effective size    946
top-1 share       10.42%
novel rows        5,550  (absent from the pretraining vocabulary)

flavours (24):
  taste: sweet, bitter, sour, salty, umami
  odour: fruity, green, floral, fatty, woody, spicy, roasted, sulfurous, earthy, nutty, herbal, medicinal, citrus, dairy, alcoholic, meaty, minty
  other: odorless, unknown


## Function Definitions for Lead Optimization

In [2]:
def name_to_smiles(name, isomeric=True):
    results = pcp.get_compounds(name, namespace="name")
    if not results:
        return None
    c = results[0]  # first hit is usually the best match
    if isomeric:
        # .isomeric_smiles is deprecated in pubchempy 1.0.5; .smiles is the
        # replacement. Note PubChem's first hit is often the UNSPECIFIED
        # stereoisomer -- "citronellol" returns CC(CCC=C(C)C)CCO with no
        # stereocentre, though the (R) and (S) forms smell different. Pass a
        # SMILES directly when the stereochemistry matters.
        return getattr(c, "smiles", None) or c.isomeric_smiles
    # PubChem renamed CanonicalSMILES → ConnectivitySMILES in 2025;
    # try the new property first, then fall back.
    return c.to_dict().get("connectivity_smiles") or c.canonical_smiles

def lead_optimization_pocketless(
    input_compound:   str | Chem.Mol,      # SMILES string or RDKit Mol
    lead_optimizer:   LeadOptimizer,
    flavor_condition: list[str],           # from PERMISSIBLE_FLAVORS
    top_k:            int = 10,
    max_decompositions: int = 4,
    retro:            bool = False,   # AIZynthFinder route per compound
    retro_timeout:    int = 3600,
    gallery:          Optional[str | Path] = "lead_optimization.pdf",
    panel:            bool = False,  # dock into the receptors for this percept
):
    """Flavour-conditioned lead optimization, scored and rendered.

    Returns a LeadOptimizationReport:

        .table      deduplicated products, best score first, with
                    retrieval_score / MW / logP / QED / SAScore / NPScore,
                    plus d<prop> = product - input for each of those
        .reference  the INPUT compound's own specs, same five properties
        .compounds  the product SMILES, in table order
        .failures   suggestions that could not be assembled, WITH the reason
        .gallery    path to the PDF/PNG, or None if drawing was unavailable
        .results    the raw per-slot SlotResult list

    With `panel=True` the table also carries `vina_<receptor>` columns -- every
    product docked into the HUMAN receptors that mediate `flavor_condition`
    (T2R for bitter, TRPM8 for minty, TRPV1/TRPA1 for spicy, and so on) -- plus
    `panel_best` and `panel_best_receptor`. Docking never sees the model, so
    those numbers are free to disagree with `retrieval_score`; that is the only
    reason they are worth computing. Each receptor is validated by redocking its
    own crystal ligand at median <= 2.5 A over three seeds.

    `retrieval_score` is the logQ-corrected value that actually ranks
    (sim/tau + log p(k)), not a similarity -- do not re-sort the table by
    anything else and expect the model's ordering back.

    Products are keyed on canonical SMILES across every decomposition and slot,
    so the same molecule proposed from three slots is ONE compound with
    n_slots=3, rather than three.

    `retro=True` adds an AIZynthFinder route search per compound
    (retro_solved / retro_steps / retro_routes / retro_materials), and the full
    route tree lands on `.routes` for the HTML viewer. Slow -- seconds to
    minutes per molecule -- so it is off by default.

    `max_decompositions` is how many ways the input is cut into core + R-groups
    before any retrieval happens. It is the main control on how much comes back:
    the work is roughly max_decompositions x slots-per-decomposition x top_k, so
    raising it explores more of the molecule and costs proportionally. 1 uses
    only the highest-ranked decomposition.
    """
    return run_lead_optimization(
        input_compound, lead_optimizer, flavor_condition,
        top_k=top_k, max_decompositions=max_decompositions,
        retro=retro, retro_timeout=retro_timeout,
        pocket_condition=None, gallery=gallery,
        panel=panel,
    )


def lead_optimization_pocket(
    input_compound:   str | Chem.Mol,
    lead_optimizer:   LeadOptimizer,
    flavor_condition: list[str],
    pocket_condition: np.ndarray,          # 1280-d ESM-2 pocket embedding
    receptor:         Optional[str | Path] = None,   # structure to DOCK into
    dock:             bool = True,
    ligand_resname:   Optional[str] = None,          # crystal ligand, for the box
    ligand_resname_smiles: Optional[str] = None,     # its SMILES, for the control
    exhaustiveness:   int = 8,
    retro:            bool = False,        # AIZynthFinder route search
    retro_timeout:    int = 3600,
    top_k:            int = 10,
    max_decompositions: int = 4,
    gallery:          Optional[str | Path] = "lead_optimization_pocket.pdf",
    pose_dir:         Optional[str | Path] = "docked_poses",
):
    """The same, additionally conditioned on a pocket.

    Get `pocket_condition` from a structure with

        pocket_embedding_from_structure(Path("receptor.cif"), device="cuda")

    Calculate the vina-related scores additionally, and also render the docking poses (if possible)

    retro=True additionally runs an AIZynthFinder route search per product,
    adding retro_solved / retro_steps / retro_score. Stronger than SAScore,
    which is a fragment heuristic that never attempts a synthesis -- the two
    disagree most on glycosylations, which this pipeline proposes often.

    SLOW: seconds to minutes per molecule, so it is off by default and absent
    from training entirely. It runs in its own environment because
    AIZynthFinder pins rdkit<2024 while this project needs rdkit>=2024.3.1 --
    mutually exclusive, and rdkit 2023.x would shift the enum indices the
    corpus stores.

    retro_solved means a route to purchasable stock was found within the search
    budget. Not a claim a chemist would run it, and False conflates "no route"
    with "none found in time".
    """
    return run_lead_optimization(
        input_compound, lead_optimizer, flavor_condition,
        top_k=top_k, max_decompositions=max_decompositions,
        pocket_condition=pocket_condition, gallery=gallery,
        receptor=receptor, dock=dock, exhaustiveness=exhaustiveness,
        retro=retro, retro_timeout=retro_timeout,
        ligand_resname=ligand_resname,
        ligand_resname_smiles=ligand_resname_smiles,
        pose_dir=pose_dir,
    )


## Run (Pocket-less Flavor Lead Optimization)

In [3]:
COMPOUND_NAME     = "citronellol"
FLAVOR_CONDITION  = ["sour", "minty"]
#########################################################################################################
COMPOUND_SMILES   = name_to_smiles(COMPOUND_NAME)
GALLERY_NAME      = f"{COMPOUND_NAME}_{'+'.join(FLAVOR_CONDITION)}.pdf"
VIEWER_HTML_NAME  = f"{COMPOUND_NAME}_{'+'.join(FLAVOR_CONDITION)}_pocketless.html"
VIEWER_HTML_TITLE = f"Lead Optimization > {COMPOUND_NAME} > {'+'.join(FLAVOR_CONDITION)} > (pocket-less)"
#########################################################################################################

report = lead_optimization_pocketless(
    COMPOUND_SMILES, opt, FLAVOR_CONDITION,
    top_k=20,
    panel=True,          # + vina_<receptor> columns for this percept

    max_decompositions=4,
    gallery=GALLERY_NAME,
    retro=True,
    retro_timeout=3600,
)
print(report)
print(f"gallery: {report.gallery}")
if len(report.failures):
    print(f"{len(report.failures)} suggestion(s) could not be assembled")

print("\nINPUT", report.input_smiles)
for k, v in report.reference.items():
    print(f"   {k:<8} {v:.3f}" if v is not None else f"   {k:<8} n/a")

# display(), not a bare expression: only the LAST expression in a cell renders,
# and the viewer comes after this one.
display(report.table[["product", "rgroup", "retrieval_score",
                      "MW", "dMW", "logP", "dlogP", "QED", "dQED",
                      "SAScore", "dSAScore", "NPScore", "dNPScore",
                      "retro_solved", "retro_steps",
                      "is_novel", "n_slots", "is_input"]])

viewer = render_html(report, VIEWER_HTML_NAME, title=VIEWER_HTML_TITLE)
print(f"viewer:  {viewer}")
display(IFrame(str(viewer), width="100%", height=620))

reading NP model ...
model in


<LeadOptimizationReport 'CC(C)=CCCC(C)CCO' flavor=['sour', 'minty'] 38 compounds>
gallery: citronellol_sour+minty.pdf
1 suggestion(s) could not be assembled

INPUT CC(C)=CCCC(C)CCO
   MW       156.269
   logP     2.751
   QED      0.607
   SAScore  2.899
   NPScore  2.504
   retro_solved 1.000
   retro_steps 1.000


,product,rgroup,retrieval_score,MW,dMW,logP,dlogP,QED,dQED,SAScore,dSAScore,NPScore,dNPScore,retro_solved,retro_steps,is_novel,n_slots,is_input
0,CC(CCO)CCCC(C)(C)O,*C(C)(C)O,49.901577,174.284,1.801500e+01,1.94610,-0.80520,0.645824,0.039077,2.877060,-0.022302,1.859495,-0.644187,True,3,False,1,False
1,C=C(C)CCCC(C)CCO,*C(=C)C,48.869865,156.269,-2.842171e-14,2.75130,0.00000,0.586191,-0.020556,2.950812,0.051450,1.975655,-0.528028,True,1,False,1,False
2,CCCCCCC(C)CCO,*[CH2]CC,47.478157,158.285,2.016000e+00,2.97530,0.22400,0.564680,-0.042066,2.363417,-0.535945,1.568832,-0.934850,True,2,False,1,False
3,CC(C)=CCCC(C)CC(=O)O,*[CH2]C(=O)O,46.518791,170.252,1.398300e+01,2.84360,0.09230,0.644037,0.037290,2.788341,-0.111020,1.942440,-0.561242,True,1,False,1,False
4,CC(C)=CCCC(C)=CC(=O)O,*[CH]C(=O)O,45.071617,168.236,1.196700e+01,2.76370,0.01240,0.517363,-0.089384,2.548391,-0.350971,2.422719,-0.080963,True,2,False,1,False
5,CC(C)CCCCC(C)CCO,*[CH2]C(C)C,44.882877,172.312,1.604300e+01,3.22130,0.47000,0.584544,-0.022203,2.494573,-0.404788,1.382581,-1.121101,True,5,False,1,False
6,CC(C)=CCCC(C)=CCCC(C)CCO,*C(C)CCC=C(C)C,44.674980,224.388,6.811900e+01,4.47780,1.72650,0.602745,-0.004001,3.095334,0.195972,2.550030,0.046348,False,6,False,1,False
7,CC(=CCCC(C)CCO)CO,*C(C)CO,44.409069,172.268,1.599900e+01,1.72370,-1.02760,0.598880,-0.007867,3.158336,0.258975,2.794230,0.290548,True,2,False,1,False
8,CC(CCO)CCCC(C)(C)C,*C(C)(C)C,44.034626,172.312,1.604300e+01,3.22130,0.47000,0.674798,0.068051,2.765696,-0.133666,1.089282,-1.414400,True,5,False,1,False
9,CC(=CCCC(C)CCO)C(=O)O,*C(C)C(=O)O,43.931541,186.251,2.998200e+01,1.81600,-0.93530,0.621915,0.015169,3.013488,0.114126,2.552372,0.048690,True,1,False,1,False


viewer:  citronellol_sour+minty_pocketless.html


## Run (Pocket-aware Flavor Lead Optimization)

`molplatte-final` has an UNTRAINED pocket path, so a pocket contributes exactly
zero to the RANKING here. That is deliberate and now well characterised: pocket
conditioning was tested at four capacity levels, two learning rates and two
initialisation scales, each against a permuted-pocket control, and every
comparison came back null. A model trained on real pockets and one trained on
randomly permuted pockets reach identical validation loss to four decimals.

`s3-pocket-supervised-s911012_best.pt` does have a live pocket path, but it was
trained with the recipe since shown to be actively harmful (lr 1e-3 with the
projectors unfrozen: -0.0543 H@1, -7.3 sigma). Use it only to demonstrate the
live path, not for results.

The receptor still earns its keep in this section -- it drives DOCKING and the
pose rendering, both independent of the model. See
`docs/pocket_retrieval_baseline_2026-09-19.md` for the one pocket signal that
does work (nearest-pocket R-group priors, useful at top-1..5 only).

`dock=True` docks every product into the receptor with AutoDock Vina, adding
`vina_score` (kcal/mol, lower is better) and `dvina` against the input. Docking
is the first check here INDEPENDENT of the model, so it is free to disagree
with `retrieval_score`.

The crystal ligand is redocked first and `.redock_rmsd` reports how far it
lands from its own known pose -- under ~2 A means the receptor prep, protonation
and box reproduce a known answer. A docking setup fails silently otherwise.

Retrieval caveat unchanged: pocket conditioning measured null on held-out
receptors (`docs/step3_pocket_capacity_2026-09-09.md`). The pocket moving the
ranking is not the same as moving it correctly on an unseen receptor.


In [3]:
PDB               = "9W0U"          # bitter T2R receptor
CCD               = "A1EUI"         # its crystal ligand, a polymethoxyflavone
FLAVOR_CONDITION  = ["bitter"]

# WHICH CHECKPOINT FOR THE POCKET SECTION -- a real trade-off, not a default.
#
#   molplatte-final-v2-w512     H@1 0.3648, pocket path UNTRAINED (inert)
#   s3-pocket-supervised        H@1 ~0.325 at width 300, pocket path LIVE
#
# v2 is the better model by every retrieval measure, and its pocket contributes
# exactly nothing -- the report says so loudly.
#
# The -0.0763 "pocket conditioning is harmful" figure once quoted here was a
# CONFOUND: that run left 2.1M params adapting to 269 records at the sweep's
# worst learning rate, so it measured catastrophic forgetting, not pockets.
# Freezing the flavour pathway removes the loss entirely (0.0000 on all five
# folds) -- and leaves pocket conditioning INERT rather than useful.
# See docs/pocket_forgetting_2026-09-17.md.
#
# So v2 is the default: the receptor still drives DOCKING and the pose renders,
# which is where the structure genuinely earns its keep, while the conditioning
# -- the part that does not work -- is honestly reported as inert. Swap the
# line below to see the live-pocket path instead.
POCKET_CKPT       = Path.home()/"checkpoints"/"molplatte"/"molplatte-final-v2-w512.pt"
# POCKET_CKPT     = Path.home()/"checkpoints"/"molplatte"/"s3-pocket-supervised-s911012_best.pt"
STRUCTURE         = Path.home()/"datasets"/"tastepocket"/"structures"/"cif"/f"{PDB}.cif"

TAG               = f"{PDB}_{CCD}_{'+'.join(FLAVOR_CONDITION)}"
GALLERY_NAME      = f"{TAG}_pocket-aware.pdf"
POSE_DIR          = f"{TAG}_poses"
VIEWER_HTML_NAME  = f"{TAG}_pocket-aware.html"
VIEWER_HTML_TITLE = f"Lead Optimization > {PDB} ({CCD}) > {'+'.join(FLAVOR_CONDITION)} > (pocket-aware)"
###################################################################

# The input compound is the complex's OWN crystal ligand: the pocket and the
# molecule must come from the same structure, or the conditioning describes a
# site this compound was never in.
COMPOUND_SMILES = next(
    torch.load(f, weights_only=False)["smiles"]
    for f in glob.glob(str(DATA/"tastepocket_corpus"/"naveja_recap"/"*"/"*.pt"))
    if torch.load(f, weights_only=False)["meta"].get("ccd") == CCD)

opt_pocket = LeadOptimizer.load(POCKET_CKPT, VOCAB, device="cuda")
opt_pocket.build_library(batch_size=1024)
pocket = pocket_embedding_from_structure(STRUCTURE, device="cuda")

report_pocket = lead_optimization_pocket(
    COMPOUND_SMILES, opt_pocket, FLAVOR_CONDITION,
    pocket_condition=pocket,
    receptor=STRUCTURE, dock=True,
    ligand_resname=CCD, ligand_resname_smiles=COMPOUND_SMILES,
    retro=True, retro_timeout=3600,
    top_k=20,
    max_decompositions=4,
    gallery=GALLERY_NAME,
    pose_dir=POSE_DIR,
)
print(report_pocket)
print(f"gallery: {report_pocket.gallery}")
if len(report_pocket.failures):
    print(f"{len(report_pocket.failures)} suggestion(s) could not be assembled")

# Did the pocket do anything? Reported rather than assumed.
print("pocket changed the ranking:", report_pocket.pocket_changed_ranking)

# The redock control is what makes the vina numbers usable: it re-docks the
# CRYSTAL ligand and measures how far it lands from its own known pose.
if report_pocket.redock_rmsd is None:
    print("redock control: NOT RUN -- docking was skipped or the control failed")
else:
    verdict = "PASS" if report_pocket.redock_rmsd < 2 else "SUSPECT"
    print(f"redock control: {report_pocket.redock_rmsd:.2f} A ({verdict})")

print("\nINPUT", report_pocket.input_smiles)
for k, v in report_pocket.reference.items():
    print(f"   {k:<13} {v:.3f}" if isinstance(v, float) else f"   {k:<13} {v}")

# vina and retro are INDEPENDENT of the model and of each other -- a compound
# can dock better than the input and still have no synthetic route.
display(report_pocket.table[["product", "rgroup", "retrieval_score",
                             "vina_score", "dvina",
                             "retro_solved", "retro_steps", "retro_n_materials",
                             "MW", "dMW", "QED", "dQED", "SAScore", "dSAScore",
                             "is_novel", "n_slots"]])

# PyMOL writes <pose>.render.png beside each SDF; pass them so every row in the
# viewer can show its own docked pose.
poses = {}
if report_pocket.pose_dir:
    targets = sorted({s.product for sl in report_pocket.results
                      for s in sl.suggestions if s.product}
                     | {report_pocket.input_smiles})
    for i, smi in enumerate(targets):
        png = Path(report_pocket.pose_dir) / f"pose_{i:03d}.render.png"
        if png.is_file():
            poses[smi] = png

viewer_pocket = render_html(report_pocket, VIEWER_HTML_NAME,
                            title=VIEWER_HTML_TITLE, pose_pngs=poses)
print(f"viewer:  {viewer_pocket}   ({len(poses)} poses embedded)")
display(IFrame(str(viewer_pocket), width="100%", height=620))


Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_1283481/876658959.py:107: RuntimeWarning: POCKET HAD NO EFFECT: the ranking is identical with and without it. Here this checkpoint's pocket reduction is UNTRAINED (zero-init), so the pocket contributes exactly zero. Treat this result as flavour-conditioned only.
  return run_lead_optimization(
reading NP model ...
model in


<LeadOptimizationReport 'COc1ccc(-c2cc(=O)c3c(OC)c(OC)c(OC)c(OC)c3o2)cc1' flavor=['bitter'] 58 compounds, pocket INERT, redock 1.58A>
gallery: 9W0U_A1EUI_bitter_pocket-aware.pdf
pocket changed the ranking: False
redock control: 1.58 A (PASS)

INPUT COc1ccc(-c2cc(=O)c3c(OC)c(OC)c(OC)c(OC)c3o2)cc1
   MW            372.373
   logP          3.503
   QED           0.655
   SAScore       2.374
   NPScore       0.831
   vina_score    -7.827
   retro_solved  True
   retro_steps   2


,product,rgroup,retrieval_score,vina_score,dvina,retro_solved,retro_steps,retro_n_materials,MW,dMW,QED,dQED,SAScore,dSAScore,is_novel,n_slots
0,COc1ccc(-c2cc(=O)c3c(OC)c(OC)c(OC)c(OC)c3o2)cc1,*OC,51.937485,-7.827,0.000,True,2,3,372.373,0.000,0.655289,0.000000,2.374225,0.000000,False,3
1,COc1c(OC)c(OC)c2c(=O)cc(-c3ccc(O)cc3)oc2c1OC,*c1ccc(O)cc1,51.473106,-7.683,0.144,True,2,4,358.346,-14.027,0.749188,0.093900,2.463541,0.089316,False,1
2,COc1ccc(-c2cc(=O)c3c(O[C@H]4O[C@H](CO)[C@@H](O...,*O[C@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1O,49.739491,-7.936,-0.109,False,3,3,520.487,148.114,0.328389,-0.326900,3.986661,1.612436,False,1
3,COc1c(OC)c(OC)c2c(=O)cc(-c3ccc(O)c(O)c3)oc2c1OC,*c1ccc(O)c(O)c1,49.387074,-7.747,0.080,True,2,4,374.345,1.972,0.656590,0.001301,2.601827,0.227602,False,1
4,COc1cc(-c2cc(=O)c3c(OC)c(OC)c(OC)c(OC)c3o2)ccc1O,*c1ccc(O)c(OC)c1,49.112770,-7.846,-0.019,True,2,4,388.372,15.999,0.687634,0.032345,2.536784,0.162559,False,1
5,COc1ccc(-c2cc(=O)c3c(OC)c(OC)c(OC)c(OC)c3o2)cc1OC,*c1ccc(OC)c(OC)c1,47.987881,-7.633,0.194,True,2,3,402.399,30.026,0.593530,-0.061759,2.450474,0.076249,False,1
6,COc1ccc(-c2cc(=O)c3c(OC)c(OC)c(OC)c(O[C@H]4O[C...,*O[C@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1O,47.500565,-7.759,0.068,False,3,3,520.487,148.114,0.328389,-0.326900,3.978489,1.604264,False,1
7,COc1c(OC)c(OC)c2c(=O)cc(-c3ccc4c(c3)OCO4)oc2c1OC,*c1ccc2c(c1)OCO2,47.374218,-8.107,-0.280,True,2,3,386.356,13.983,0.660760,0.005471,2.587031,0.212806,False,1
8,COc1ccccc1-c1cc(=O)c2c(OC)c(OC)c(OC)c(OC)c2o1,*c1ccccc1OC,47.131260,-7.396,0.431,True,2,3,372.373,0.000,0.655289,0.000000,2.424007,0.049782,False,1
9,COc1c(OC)c(OC)c2c(=O)cc(-c3ccccc3O)oc2c1OC,*c1ccccc1O,47.108891,-7.666,0.161,True,2,3,358.346,-14.027,0.749188,0.093900,2.531939,0.157714,False,1


viewer:  9W0U_A1EUI_bitter_pocket-aware.html   (6 poses embedded)
